# NetShield AI - Experiment 2: Generalization (Zero-Shot Detection)
This experiment evaluates whether the model has learned true 'malicious anomalies' or just memorized flow signatures.
We train the model exclusively on **Monday, Tuesday, and Wednesday** traffic (Benign, Brute Force, DoS, Heartbleed).
We test the model exclusively on **Thursday and Friday** traffic (Web Attacks, Infiltration, Botnet, PortScan, DDoS).
Because the test set contains attack families the model has **never seen before**, this tests true Zero-Shot malicious generalization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

plt.style.use('dark_background')
sns.set_palette('muted')

## 1. Load Data with Day/Scenario Splitting

In [ ]:
dataset_dir = r'D:\Infosys\2017_improved'
all_files = glob.glob(os.path.join(dataset_dir, '*.csv'))

train_dfs = []
test_dfs = []

for file in all_files:
    filename = os.path.basename(file).lower()
    print(f'Loading {filename}...')
    temp_df = pd.read_csv(file)
    
    # Split by Scenario (Day)
    if 'monday' in filename or 'tuesday' in filename or 'wednesday' in filename:
        train_dfs.append(temp_df)
    elif 'thursday' in filename or 'friday' in filename:
        test_dfs.append(temp_df)

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)

df_train.columns = df_train.columns.str.strip().str.lower().str.replace(' ', '_')
df_test.columns = df_test.columns.str.strip().str.lower().str.replace(' ', '_')

print(f'\nScenario Train Shape (Mon-Wed): {df_train.shape}')
print(f'Scenario Test Shape (Thu-Fri): {df_test.shape}')

## 2. Cleaning & Binarization

In [ ]:
feature_drop = [
    'label',
    'attempted_category',
    'id',
    'timestamp',
    'flow_id',
    'src_ip',
    'dst_ip',
    'src_port',
    'dst_port',
    'protocol'
]

def preprocess_scenario(df, is_train=False):
    # Extract Labels and Binarize (0 = BENIGN, 1 = MALICIOUS)
    y = (df['label'] != 'BENIGN').astype(int)
    
    # Drop network artifacts to prevent flow-memorization
    X = df.drop(columns=[col for col in feature_drop if col in df.columns], errors='ignore')
    X = X.select_dtypes(include=[np.number]).fillna(0)
    
    # Combine for deduplication
    temp = X.copy()
    temp['label_bin'] = y
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna()
    temp = temp.drop_duplicates()
    
    if is_train:
        # Downsample BENIGN in training only
        benign = temp[temp['label_bin'] == 0]
        malicious = temp[temp['label_bin'] == 1]
        if len(benign) > 200000:
            benign = benign.sample(n=200000, random_state=42, replace=False)
        temp = pd.concat([benign, malicious]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    y_out = temp['label_bin']
    X_out = temp.drop(columns=['label_bin'])
    return X_out, y_out

print("Processing Training Set (Mon-Wed)...")
X_train, y_train = preprocess_scenario(df_train, is_train=True)

print("Processing Untouched Test Set (Thu-Fri)...")
X_test, y_test = preprocess_scenario(df_test, is_train=False)

print(f"\nFinal Train Shape: {X_train.shape}")
print(f"Final Test Shape: {X_test.shape} (Zero-Shot Generalization targets)")

## 3. Train Binary Generalization Model

In [ ]:
print("Training Random Forest on Days 1-3...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)
print("Predicting on unseen Days 4-5...")
y_pred = model.predict(X_test)
print("Done!")

## 4. Generalization Evaluation

In [ ]:
from sklearn.metrics import classification_report

print("="*60)
print("🛡️ NetShield AI - Zero-Shot Generalization Experiment 🛡️")
print("="*60)
print("Training: Brute Force, DoS, Heartbleed")
print("Testing: Web Attacks, Infiltration, Botnet, PortScan, DDoS\n")

print("📊 STANDARD ML CLASSIFICATION REPORT (Precision/Recall/F1) 📊")
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'MALICIOUS']))
print("\n")

binary_cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = binary_cm.ravel()

bin_TDR = TP / (TP + FN) if (TP + FN) > 0 else 0
bin_FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
bin_FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

print(f"Malicious Traffic TDR: {bin_TDR*100:.2f}%")
print(f"Malicious Traffic FPR: {bin_FPR*100:.4f}%")
print(f"Malicious Traffic FNR: {bin_FNR*100:.4f}%")

plt.figure(figsize=(8, 6))
sns.heatmap(binary_cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Predicted BENIGN', 'Predicted MALICIOUS'], 
            yticklabels=['Actual BENIGN', 'Actual MALICIOUS (Unseen Attacks)'])
plt.title('NetShield AI - Scenario Generalization Matrix')
plt.show()